# Qdrant dense + BGE reranker

Questo notebook aggiunge un secondo stadio alla baseline già validata:

> `D_full_enriched` → BGE-M3 dense su Qdrant → `bge-reranker-v2-m3` → top-k finale.

Presupposto: il notebook `05_qdrant_dense_integration.ipynb` è terminato con equivalenza NumPy ↔ Qdrant. La collection **non viene ricreata né reindicizzata**: il reranker usa a runtime il campo `document` già presente nel payload.

L'obiettivo è verificare se il reranker migliora la qualità rispetto a `D_full_enriched/dense`, scegliere quanti candidati passargli e fissare l'API che verrà poi esposta tramite MCP.


## Struttura

1. configurazione e controllo della collection esistente;
2. caricamento lazy di BGE-M3 e del cross-encoder;
3. esempio ispezionabile di dense retrieval e reranking;
4. confronto su tutte le query con candidate pool 10, 20 e 30;
5. controllo delle invarianti: provenienza dei candidati, filtri e ranking;
6. NDCG/Recall/Precision e latenza;
7. gate decisionale prima di MCP e LLM.

BM25, hybrid, MCP e LLM rimangono fuori da questo esperimento. Il retriever candidato è volutamente quello risultato migliore: dense su `D_full_enriched`.


## 0. Prerequisiti

Dalla root del progetto:

```bash
docker compose up -d
python -m pip install -e ".[bge]"
```

Il primo utilizzo scarica `BAAI/bge-reranker-v2-m3`. `use_fp16` viene abilitato automaticamente quando è disponibile CUDA.


In [ ]:
# Se necessario, eseguire una volta dalla root del progetto:
# %pip install -e ".[bge]"


## 1. Import e individuazione del progetto

Il notebook orchestra classi definite in `src/movie_rag`; non ridefinisce la pipeline. Questo evita che notebook, futuro server MCP e applicazione usino implementazioni diverse.


In [1]:
from __future__ import annotations

import importlib.metadata as importlib_metadata
import json
import os
import sys
import warnings
from pathlib import Path
from time import perf_counter

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

import numpy as np
import pandas as pd
from IPython.display import display


def locate_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "movie_rag").is_dir():
            return candidate
    raise RuntimeError("Non trovo src/movie_rag: avviare il notebook dalla cartella del progetto.")


PROJECT_ROOT = locate_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Importing modules")

from movie_rag import (
    BASELINE,
    RERANKER,
    BGEM3DenseEncoder,
    BGEV2M3Reranker,
    DenseMovieRetriever,
    RerankedMovieRetriever,
    build_annotation_extension,
    connect_qdrant,
    ensure_dense_collection,
    evaluate_results,
    rerank_candidates,
)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 100)

print("Project root:", PROJECT_ROOT)
for package in ("pandas", "numpy", "qdrant-client", "FlagEmbedding"):
    try:
        print(f"{package}: {importlib_metadata.version(package)}")
    except importlib_metadata.PackageNotFoundError:
        print(f"{package}: non installato")


Importing modules
Project root: /Users/danieleraveglia/Documents/uniMi/TESI/progetto
pandas: 3.0.5
numpy: 2.5.1
qdrant-client: 1.19.0
FlagEmbedding: 1.4.2


## 2. Configurazione dell'esperimento

Le impostazioni stabili sono in `RERANKER`; qui si dichiarano esplicitamente i valori da confrontare.

- `TOP_K=10`: dimensione della classifica finale;
- candidate pool `10/20/30`: quanti risultati dense vengono valutati dal cross-encoder;
- `max_length=1024`: lunghezza massima esplicita per le coppie query-documento;
- `normalize_scores=True`: sigmoid sui logit; non modifica l'ordine, ma produce valori 0–1.

Gli score non vanno confrontati come probabilità calibrate tra query differenti.


In [2]:
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY") or None
COLLECTION_NAME = BASELINE.collection_name

TOP_K = RERANKER.top_k
CANDIDATE_POOL_SIZES = (10, 20, 30)
DEFAULT_CANDIDATE_POOL_SIZE = RERANKER.candidate_pool_size
EXACT_SEARCH = True  # coerente con la verifica sui 50 film
SAVE_OUTPUTS = True
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "reranking_v1"

ANNOTATION_CANDIDATES = [
    PROJECT_ROOT / "notebooks" / "annotation_pool_50_movies_AA.csv",
    PROJECT_ROOT / "annotation_pool_50_movies_AA.csv",
]
ANNOTATION_PATH = next(
    (path for path in ANNOTATION_CANDIDATES if path.exists()),
    ANNOTATION_CANDIDATES[0],
)
ANNOTATION_EXTENSION_PATH = OUTPUT_DIR / "reranker_annotation_extension.csv"

assert TOP_K <= min(CANDIDATE_POOL_SIZES)
assert DEFAULT_CANDIDATE_POOL_SIZE in CANDIDATE_POOL_SIZES

display(pd.Series({
    "collection": COLLECTION_NAME,
    "dense_model": BASELINE.embedding_model,
    "reranker_model": RERANKER.model_name,
    "document": BASELINE.experiment_name,
    "candidate_pool_sizes": CANDIDATE_POOL_SIZES,
    "top_k": TOP_K,
    "reranker_batch_size": RERANKER.batch_size,
    "reranker_max_length": RERANKER.max_length,
    "normalize_scores": RERANKER.normalize_scores,
}))


collection              movies_d_full_enriched_dense_v1
dense_model                                 BAAI/bge-m3
reranker_model                  BAAI/bge-reranker-v2-m3
document                                D_full_enriched
candidate_pool_sizes                       (10, 20, 30)
top_k                                                10
reranker_batch_size                                  16
reranker_max_length                                1024
normalize_scores                                   True
dtype: object

## 3. Connessione e controllo della collection esistente

Questa sezione non modifica i punti. Verifica schema dense, numero di film e presenza del documento completo nel payload: senza quel campo il cross-encoder non potrebbe lavorare.


In [3]:
client = connect_qdrant(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout_seconds=30,
)
collection_result = ensure_dense_collection(
    client=client,
    collection_name=COLLECTION_NAME,
    recreate=False,
)
point_count = int(client.count(collection_name=COLLECTION_NAME, exact=True).count)
sample_points, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=1,
    with_payload=True,
    with_vectors=False,
)
if not sample_points:
    raise RuntimeError("La collection esiste ma non contiene film. Eseguire prima il notebook 05.")
sample_payload = dict(sample_points[0].payload or {})
if not str(sample_payload.get(RERANKER.document_payload_field, "")).strip():
    raise RuntimeError("Nel payload manca il documento D_full_enriched richiesto dal reranker.")
if sample_payload.get("representation_version") != BASELINE.representation_version:
    raise RuntimeError("La collection contiene una versione documentale diversa dalla baseline.")

print(collection_result)
print("Film nella collection:", point_count)
print("Documento payload verificato per:", sample_payload.get(BASELINE.title_column))


CollectionResult(collection_name='movies_d_full_enriched_dense_v1', created=False, vector_name='dense', vector_size=1024)
Film nella collection: 50
Documento payload verificato per: Toy Story (1995)


## 4. Modelli e pipeline riutilizzabile

`DenseMovieRetriever` recupera da Qdrant. `RerankedMovieRetriever` gli applica sopra il cross-encoder. Questa seconda classe è il confine che il futuro tool MCP potrà richiamare senza conoscere Pandas, Qdrant o FlagEmbedding.


In [4]:
encoder = BGEM3DenseEncoder(use_fp16=None)
reranker = BGEV2M3Reranker(use_fp16=None)

dense_retriever = DenseMovieRetriever(
    client=client,
    encoder=encoder,
    collection_name=COLLECTION_NAME,
)
reranked_retriever = RerankedMovieRetriever(
    dense_retriever=dense_retriever,
    reranker=reranker,
)

# Il caricamento esplicito separa il tempo di download/load dalla misura di latenza delle query.
encoder.load()
reranker.load()
print("Modelli pronti:", encoder.model_name, "+", reranker.model_name)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Modelli pronti: BAAI/bge-m3 + BAAI/bge-reranker-v2-m3


## 5. Prima query: cosa cambia davvero

Il reranker non cerca nuovi film. Può soltanto riordinare e selezionare il candidate pool dense. Per questo salviamo sempre rango e score di entrambi gli stadi.


In [5]:
EXAMPLE_QUERY = "un film malinconico sul rapporto tra memoria, identità e tempo"
EXAMPLE_FILTERS = {}

example_candidates = reranked_retriever.retrieve_candidates(
    query=EXAMPLE_QUERY,
    filters=EXAMPLE_FILTERS,
    candidate_pool_size=DEFAULT_CANDIDATE_POOL_SIZE,
    exact=EXACT_SEARCH,
)
example_reranked = rerank_candidates(
    query=EXAMPLE_QUERY,
    candidates=example_candidates,
    reranker=reranker,
    top_k=TOP_K,
)

candidate_ids = set(example_candidates[BASELINE.id_column])
assert set(example_reranked[BASELINE.id_column]) <= candidate_ids
assert example_reranked["rank"].tolist() == list(range(1, len(example_reranked) + 1))

columns = [
    "rank", BASELINE.id_column, BASELINE.title_column,
    "reranker_score", "dense_rank", "dense_score",
]
display(example_reranked[[column for column in columns if column in example_reranked]])


Compute Scores: 100%|██████████| 2/2 [00:03<00:00,  1.99s/it]


,rank,movieId,canonical_title,reranker_score,dense_rank,dense_score
0,1,2959,Fight Club (1999),0.021775,3,0.515896
1,2,150,Apollo 13 (1995),0.020382,12,0.471546
2,3,110,Braveheart (1995),0.019399,18,0.464727
3,4,2028,Saving Private Ryan (1998),0.010773,17,0.469145
4,5,2571,"Matrix, The (1999)",0.010551,5,0.506349
5,6,4226,Memento (2000),0.009557,1,0.531711
6,7,1270,Back to the Future (1985),0.008932,4,0.509704
7,8,1580,Men in Black (a.k.a. MIB) (1997),0.007402,19,0.460466
8,9,590,Dances with Wolves (1990),0.005866,9,0.481769
9,10,2858,American Beauty (1999),0.005452,10,0.474251


## 6. Query set

Riutilizziamo esattamente le dieci query della baseline Qdrant, compresi i filtri. Il reranking avviene **dopo** il filtro: un film escluso non può riapparire.


In [17]:
TEST_QUERIES = pd.DataFrame(
    [
        {"query_id": "Q01", "category": "semantic_theme_it", "query": "un film malinconico sul rapporto tra memoria, identità e tempo", "filters": {}},
        {"query_id": "Q02", "category": "semantic_theme_en", "query": "a melancholic film about memory, identity and time", "filters": {}},
        {"query_id": "Q03", "category": "mood", "query": "qualcosa di emozionante, ottimista e commovente", "filters": {}},
        {"query_id": "Q04", "category": "named_entity_cast", "query": "un film con Tom Hanks", "filters": {}},
        {"query_id": "Q05", "category": "named_entity_director", "query": "un film diretto da Christopher Nolan", "filters": {}},
        {"query_id": "Q06", "category": "collection", "query": "un film appartenente a una saga o collection famosa", "filters": {}},
        {"query_id": "Q07", "category": "mixed_constraints", "query": "un film di fantascienza riflessivo sotto le due ore", "filters": {"max_runtime": 120, "include_genres": ["Science Fiction", "Sci-Fi"]}},
        {"query_id": "Q08", "category": "exclusion", "query": "un film teso ma non horror", "filters": {"exclude_genres": ["Horror"]}},
        {"query_id": "Q09", "category": "year_constraint", "query": "un dramma moderno sulle relazioni familiari", "filters": {"min_year": 2000}},
        # {"query_id": "Q10", "category": "country_language", "query": "un film italiano su famiglia e cambiamento sociale", "filters": {"countries": ["IT"], "languages": ["it"]}},
    ]
)
display(TEST_QUERIES)


,query_id,category,query,filters
0,Q01,semantic_theme_it,"un film malinconico sul rapporto tra memoria, identità e tempo",{}
1,Q02,semantic_theme_en,"a melancholic film about memory, identity and time",{}
2,Q03,mood,"qualcosa di emozionante, ottimista e commovente",{}
3,Q04,named_entity_cast,un film con Tom Hanks,{}
4,Q05,named_entity_director,un film diretto da Christopher Nolan,{}
5,Q06,collection,un film appartenente a una saga o collection famosa,{}
6,Q07,mixed_constraints,un film di fantascienza riflessivo sotto le due ore,"{'max_runtime': 120, 'include_genres': ['Science Fiction', 'Sci-Fi']}"
7,Q08,exclusion,un film teso ma non horror,{'exclude_genres': ['Horror']}
8,Q09,year_constraint,un dramma moderno sulle relazioni familiari,{'min_year': 2000}


## 7. Esecuzione dense e dense + reranker

Per ogni query salviamo:

- baseline dense top-10;
- candidate ranking dense per ogni `top_n`;
- classifica finale dopo reranking;
- tempo del retrieval e del solo reranking.

La latenza del primo caricamento dei modelli è esclusa.


In [18]:
def ranking_records(
    frame: pd.DataFrame,
    query_row: dict,
    system: str,
    stage: str,
    candidate_pool_size: int,
) -> list[dict]:
    records = []
    for result in frame.to_dict("records"):
        records.append({
            "query_id": query_row["query_id"],
            "query": query_row["query"],
            "category": query_row["category"],
            "experiment": BASELINE.experiment_name,
            "system": system,
            "stage": stage,
            "candidate_pool_size": candidate_pool_size,
            "rank": int(result["rank"]),
            BASELINE.id_column: result[BASELINE.id_column],
            BASELINE.title_column: result.get(BASELINE.title_column),
            "score": float(result["score"]),
            "dense_rank": result.get("dense_rank"),
            "dense_score": result.get("dense_score", result.get("score")),
            "reranker_score": result.get("reranker_score"),
        })
    return records


run_records = []
latency_records = []
integrity_records = []

for query_row in TEST_QUERIES.to_dict("records"):
    started = perf_counter()
    dense_top_k = dense_retriever.search(
        query=query_row["query"],
        filters=query_row["filters"],
        top_k=TOP_K,
        exact=EXACT_SEARCH,
    )
    dense_ms = (perf_counter() - started) * 1000
    run_records.extend(ranking_records(
        dense_top_k, query_row, "dense_qdrant", "final", TOP_K
    ))
    latency_records.append({
        "query_id": query_row["query_id"], "system": "dense_qdrant",
        "candidate_pool_size": TOP_K, "retrieval_ms": dense_ms,
        "rerank_ms": 0.0, "total_ms": dense_ms,
    })

    for pool_size in CANDIDATE_POOL_SIZES:
        started = perf_counter()
        candidates = reranked_retriever.retrieve_candidates(
            query=query_row["query"],
            filters=query_row["filters"],
            candidate_pool_size=pool_size,
            exact=EXACT_SEARCH,
        )
        retrieval_ms = (perf_counter() - started) * 1000

        started = perf_counter()
        reranked = rerank_candidates(
            query=query_row["query"],
            candidates=candidates,
            reranker=reranker,
            top_k=TOP_K,
        )
        rerank_ms = (perf_counter() - started) * 1000

        candidate_ids = set(candidates.get(BASELINE.id_column, []))
        final_ids = set(reranked.get(BASELINE.id_column, []))
        ranks_ok = reranked["rank"].tolist() == list(range(1, len(reranked) + 1))
        provenance_ok = final_ids <= candidate_ids
        if not (ranks_ok and provenance_ok):
            raise AssertionError(f"Invariante reranking fallita per {query_row['query_id']} / {pool_size}")

        candidate_system = f"dense_candidates@{pool_size}"
        reranked_system = f"dense_qdrant+reranker@{pool_size}"
        run_records.extend(ranking_records(
            candidates, query_row, candidate_system, "candidate", pool_size
        ))
        run_records.extend(ranking_records(
            reranked, query_row, reranked_system, "final", pool_size
        ))
        latency_records.append({
            "query_id": query_row["query_id"], "system": reranked_system,
            "candidate_pool_size": pool_size, "retrieval_ms": retrieval_ms,
            "rerank_ms": rerank_ms, "total_ms": retrieval_ms + rerank_ms,
        })
        integrity_records.append({
            "query_id": query_row["query_id"],
            "candidate_pool_size": pool_size,
            "candidate_count": len(candidates),
            "final_count": len(reranked),
            "provenance_ok": provenance_ok,
            "ranks_ok": ranks_ok,
        })

all_results = pd.DataFrame(run_records)
latencies = pd.DataFrame(latency_records)
integrity = pd.DataFrame(integrity_records)

print("Righe di ranking:", len(all_results))
display(integrity)
display(latencies.groupby(["system", "candidate_pool_size"], as_index=False)[["retrieval_ms", "rerank_ms", "total_ms"]].mean())


Compute Scores: 100%|██████████| 2/2 [00:04<00:00,  2.27s/it]


Righe di ranking: 777


,query_id,candidate_pool_size,candidate_count,final_count,provenance_ok,ranks_ok
0,Q01,10,10,10,True,True
1,Q01,20,20,10,True,True
2,Q01,30,30,10,True,True
3,Q02,10,10,10,True,True
4,Q02,20,20,10,True,True
5,Q02,30,30,10,True,True
6,Q03,10,10,10,True,True
7,Q03,20,20,10,True,True
8,Q03,30,30,10,True,True
9,Q04,10,10,10,True,True


,system,candidate_pool_size,retrieval_ms,rerank_ms,total_ms
0,dense_qdrant,10,34.971361,0.000000,34.971361
1,dense_qdrant+reranker@10,10,17.625574,1846.113884,1863.739458
2,dense_qdrant+reranker@20,20,24.504889,3673.024518,3697.529407
3,dense_qdrant+reranker@30,30,28.663671,3996.809940,4025.473611


## 8. Ispezione qualitativa per query

Prima delle medie, osserviamo quali film sono stati promossi o retrocessi. `dense_rank` rende visibile l'effetto del cross-encoder.


In [19]:
QUERY_TO_INSPECT = "Q01"
SYSTEMS_TO_INSPECT = ["dense_qdrant", f"dense_qdrant+reranker@{DEFAULT_CANDIDATE_POOL_SIZE}"]
inspection = all_results[
    all_results["query_id"].eq(QUERY_TO_INSPECT)
    & all_results["system"].isin(SYSTEMS_TO_INSPECT)
].sort_values(["system", "rank"])
display(inspection[[
    "query_id", "system", "rank", BASELINE.id_column, BASELINE.title_column,
    "score", "dense_rank", "dense_score", "reranker_score",
]])


,query_id,system,rank,movieId,canonical_title,score,dense_rank,dense_score,reranker_score
0,Q01,dense_qdrant,1,4226,Memento (2000),0.531711,NaN,0.531711,NaN
1,Q01,dense_qdrant,2,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),0.530614,NaN,0.530614,NaN
2,Q01,dense_qdrant,3,2959,Fight Club (1999),0.515896,NaN,0.515896,NaN
3,Q01,dense_qdrant,4,1270,Back to the Future (1985),0.509704,NaN,0.509704,NaN
4,Q01,dense_qdrant,5,2571,"Matrix, The (1999)",0.506349,NaN,0.506349,NaN
5,Q01,dense_qdrant,6,2762,"Sixth Sense, The (1999)",0.489677,NaN,0.489677,NaN
6,Q01,dense_qdrant,7,589,Terminator 2: Judgment Day (1991),0.483635,NaN,0.483635,NaN
7,Q01,dense_qdrant,8,780,Independence Day (a.k.a. ID4) (1996),0.483344,NaN,0.483344,NaN
8,Q01,dense_qdrant,9,590,Dances with Wolves (1990),0.481769,NaN,0.481769,NaN
9,Q01,dense_qdrant,10,2858,American Beauty (1999),0.474251,NaN,0.474251,NaN


## 9. Metriche con il pool annotato

Il reranker non è considerato migliore perché produce score diversi: deve migliorare NDCG@10 sullo stesso ground truth. Misuriamo anche Recall@N del candidate retriever, perché un rilevante assente dal pool non può essere recuperato dal secondo stadio.

La cella estende automaticamente il pooling con l'unione del candidate pool massimo e delle classifiche finali. I nuovi film vengono scritti in `reranker_annotation_extension.csv`: compilare `relevance` con 0/1/2 e rieseguire. Le metriche definitive restano sospese finché esistono risultati non giudicati, perché considerarli automaticamente irrilevanti penalizzerebbe il nuovo sistema.

Vengono inoltre segnalate query senza giudizi o senza risultati. Una run vuota viene valutata con metriche pari a zero, invece di sparire dalla media; nel notebook precedente le metriche salvate coprivano 9 query su 10.


In [22]:
def load_annotations(path: Path) -> pd.DataFrame:
    for separator in (";", ","):
        frame = pd.read_csv(path, sep=separator)
        if {"query_id", BASELINE.id_column, "relevance"} <= set(frame.columns):
            return frame
    raise ValueError(f"Formato annotazioni non riconosciuto: {path}")


empty_qrels = pd.DataFrame(columns=["query_id", BASELINE.id_column, "relevance"])
if "annotated_pool" in globals() and isinstance(annotated_pool, pd.DataFrame):
    base_qrels = annotated_pool.copy()
    annotation_source = "DataFrame annotated_pool in memoria"
elif ANNOTATION_PATH.exists():
    base_qrels = load_annotations(ANNOTATION_PATH)
    annotation_source = str(ANNOTATION_PATH)
else:
    base_qrels = empty_qrels.copy()
    annotation_source = None

existing_extension = (
    load_annotations(ANNOTATION_EXTENSION_PATH)
    if ANNOTATION_EXTENSION_PATH.exists()
    else empty_qrels.copy()
)
qrels = pd.concat([base_qrels, existing_extension], ignore_index=True, sort=False)
qrels = (
    qrels.dropna(subset=["relevance"])
    .drop_duplicates(["query_id", BASELINE.id_column], keep="last")
    .reset_index(drop=True)
)

final_runs = all_results[all_results["stage"].eq("final")].copy()
max_pool_system = f"dense_candidates@{max(CANDIDATE_POOL_SIZES)}"
pooling_runs = all_results[
    all_results["stage"].eq("final") | all_results["system"].eq(max_pool_system)
].copy()
new_extension = build_annotation_extension(pooling_runs, qrels)
annotation_work = pd.concat([existing_extension, new_extension], ignore_index=True, sort=False)
if not annotation_work.empty:
    annotation_work = annotation_work.drop_duplicates(
        ["query_id", BASELINE.id_column], keep="first"
    ).sort_values(["query_id", "best_rank", BASELINE.id_column], na_position="last")
unjudged_pairs = build_annotation_extension(pooling_runs, qrels)

if SAVE_OUTPUTS and not annotation_work.empty:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    annotation_work.to_csv(ANNOTATION_EXTENSION_PATH, sep=";", index=False)

final_metrics = pd.DataFrame()
candidate_metrics = pd.DataFrame()
metric_summary = pd.DataFrame()
per_query_delta = pd.DataFrame()
missing_query_judgments = sorted(
    set(TEST_QUERIES["query_id"]) - set(qrels.get("query_id", pd.Series(dtype=str)).astype(str))
)
queries_without_results = sorted(
    integrity.loc[integrity["candidate_count"].eq(0), "query_id"].unique()
)

print("Annotazioni di base:", annotation_source or "non trovate")
print("Nuove coppie query-film da giudicare:", len(unjudged_pairs))
print("Query senza alcun giudizio:", missing_query_judgments or "nessuna")
print("Query senza candidati dense:", queries_without_results or "nessuna")
if not unjudged_pairs.empty:
    display(unjudged_pairs)
    print("Compilare relevance (0/1/2) in:", ANNOTATION_EXTENSION_PATH)

evaluation_ready = bool(len(qrels)) and unjudged_pairs.empty and not missing_query_judgments
if not evaluation_ready:
    warnings.warn(
        "Metriche definitive sospese: completare il pool incrementale e rieseguire la cella."
    )
else:
    final_metrics = evaluate_results(
        final_runs,
        qrels,
        k=TOP_K,
        expected_query_ids=TEST_QUERIES["query_id"],
        strict_judgments=True,
    )
    queries_without_results = sorted(
        final_metrics.loc[final_metrics["retrieved_items_at_k"].eq(0), "query_id"].unique()
    )
    metric_columns = [f"ndcg@{TOP_K}", f"recall@{TOP_K}", f"precision@{TOP_K}"]
    metric_summary = (
        final_metrics.groupby("system", as_index=False)[metric_columns]
        .mean()
        .sort_values(f"ndcg@{TOP_K}", ascending=False)
        .reset_index(drop=True)
    )
    display(metric_summary)

    candidate_parts = []
    for pool_size in CANDIDATE_POOL_SIZES:
        system = f"dense_candidates@{pool_size}"
        subset = all_results[all_results["system"].eq(system)]
        part = evaluate_results(
            subset,
            qrels,
            k=pool_size,
            expected_query_ids=TEST_QUERIES["query_id"],
            strict_judgments=True,
        )
        part["candidate_pool_size"] = pool_size
        candidate_parts.append(part)
    candidate_metrics = pd.concat(candidate_parts, ignore_index=True)
    candidate_summary = pd.DataFrame([
        {
            "candidate_pool_size": pool_size,
            "mean_candidate_recall": candidate_metrics.loc[
                candidate_metrics["candidate_pool_size"].eq(pool_size), f"recall@{pool_size}"
            ].mean(),
        }
        for pool_size in CANDIDATE_POOL_SIZES
    ])
    display(candidate_summary)

    baseline_per_query = final_metrics[final_metrics["system"].eq("dense_qdrant")][
        ["query_id", f"ndcg@{TOP_K}"]
    ].rename(columns={f"ndcg@{TOP_K}": "dense_ndcg"})
    reranker_per_query = final_metrics[final_metrics["system"].str.startswith("dense_qdrant+reranker@")][
        ["query_id", "system", f"ndcg@{TOP_K}"]
    ].rename(columns={f"ndcg@{TOP_K}": "reranked_ndcg"})
    per_query_delta = reranker_per_query.merge(baseline_per_query, on="query_id", how="left")
    per_query_delta["delta_ndcg"] = per_query_delta["reranked_ndcg"] - per_query_delta["dense_ndcg"]
    display(per_query_delta.sort_values(["system", "delta_ndcg"]))


ParserError: Error tokenizing data. C error: Expected 1 fields in line 3, saw 2


## 10. Gate decisionale prima di MCP/LLM

Il codice MCP può essere scritto anche senza risultati definitivi, ma prima di dichiarare stabile la pipeline conviene fissare quattro elementi:

1. tutte le query hanno almeno un giudizio utilizzabile;
2. è scelto un solo `candidate_pool_size`;
3. dense + reranker migliora o almeno giustifica il costo aggiuntivo;
4. il contratto di output conserva ID, titolo, rango/score dense e score reranker.

La decisione automatica seguente è prudente: suggerisce il reranker solo se la migliore media NDCG@10 supera la baseline dense. Va comunque letta insieme ai delta per query e alla latenza.


In [ ]:
decision = {
    "pipeline_candidate": RERANKER.pipeline_name,
    "collection": COLLECTION_NAME,
    "qdrant_points": point_count,
    "integrity_checks_passed": bool(integrity[["provenance_ok", "ranks_ok"]].all().all()),
    "all_queries_judged": not missing_query_judgments,
    "missing_query_judgments": missing_query_judgments,
    "unjudged_query_movie_pairs": int(len(unjudged_pairs)),
    "evaluation_ready": evaluation_ready,
    "queries_without_results": queries_without_results,
    "recommended_candidate_pool_size": None,
    "mean_ndcg_dense": None,
    "mean_ndcg_reranked": None,
    "delta_mean_ndcg": None,
    "recommendation": "evaluate_annotations_before_mcp",
}

if not metric_summary.empty:
    dense_row = metric_summary[metric_summary["system"].eq("dense_qdrant")].iloc[0]
    reranker_rows = metric_summary[metric_summary["system"].str.startswith("dense_qdrant+reranker@")].copy()
    reranker_rows["candidate_pool_size"] = reranker_rows["system"].str.rsplit("@", n=1).str[-1].astype(int)
    best = reranker_rows.sort_values(
        [f"ndcg@{TOP_K}", "candidate_pool_size"],
        ascending=[False, True],
    ).iloc[0]
    delta = float(best[f"ndcg@{TOP_K}"] - dense_row[f"ndcg@{TOP_K}"])
    decision.update({
        "recommended_candidate_pool_size": int(best["candidate_pool_size"]),
        "mean_ndcg_dense": float(dense_row[f"ndcg@{TOP_K}"]),
        "mean_ndcg_reranked": float(best[f"ndcg@{TOP_K}"]),
        "delta_mean_ndcg": delta,
        "recommendation": "enable_reranker" if delta > 0 else "keep_dense_as_default",
    })

display(pd.Series(decision))


## 11. Salvataggio degli output

Questi file sono il checkpoint sperimentale da conservare prima di introdurre MCP. Non modificano Qdrant.


In [ ]:
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    all_results.to_csv(OUTPUT_DIR / "reranking_run_results.csv", index=False)
    latencies.to_csv(OUTPUT_DIR / "reranking_latencies.csv", index=False)
    integrity.to_csv(OUTPUT_DIR / "reranking_integrity.csv", index=False)
    if not final_metrics.empty:
        final_metrics.to_csv(OUTPUT_DIR / "reranking_metrics_at_10.csv", index=False)
        candidate_metrics.to_csv(OUTPUT_DIR / "candidate_recall_metrics.csv", index=False)
        metric_summary.to_csv(OUTPUT_DIR / "reranking_metric_summary.csv", index=False)
        per_query_delta.to_csv(OUTPUT_DIR / "reranking_per_query_delta.csv", index=False)
    with (OUTPUT_DIR / "reranking_decision.json").open("w", encoding="utf-8") as stream:
        json.dump(decision, stream, ensure_ascii=False, indent=2)
    print("Output salvati in:", OUTPUT_DIR)
else:
    print("Salvataggio disattivato.")


## 12. Confine pronto per MCP

Dopo avere scelto il candidate pool, il futuro server dovrà soltanto tradurre JSON in questa chiamata:

```python
reranked_retriever.search(
    query=user_query,
    filters=structured_filters,
    candidate_pool_size=chosen_top_n,
    top_k=10,
)
```

Non dovrà duplicare embedding, accesso a Qdrant o logica di reranking. Gli score del cross-encoder restano query-dependent e quindi non vengono salvati nella collection.

Documentazione pertinente:

- [BAAI/bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3) — coppie query-documento e `normalize=True`;
- [FlagEmbedding](https://github.com/FlagOpen/FlagEmbedding) — inference dei reranker;
- [Qdrant query API](https://qdrant.tech/documentation/concepts/search/) — candidate retrieval e payload.
